In [16]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [17]:
df = pd.read_csv("C:/Users/Yashwanth u/Desktop/Data_projects/retail-intelligence/raw/online_retail_II.csv", encoding='ISO-8859-1')

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

Shape: (1067371, 8)
Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [18]:
print("Shape:", df.shape)
print("\nColumn Names:", df.columns.tolist())
print("\nData Types:\n", df.dtypes)
print("\nNull Values:\n", df.isnull().sum())
print("\nDuplicate Rows:", df.duplicated().sum())

Shape: (1067371, 8)

Column Names: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']

Data Types:
 Invoice         object
StockCode       object
Description     object
Quantity         int64
InvoiceDate     object
Price          float64
Customer ID    float64
Country         object
dtype: object

Null Values:
 Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

Duplicate Rows: 34335


In [19]:
print("Unique Invoices:", df['Invoice'].nunique())
print("Unique StockCodes:", df['StockCode'].nunique())
print("Unique Descriptions:", df['Description'].nunique())
print("Unique Customers:", df['Customer ID'].nunique())
print("Unique Countries:", df['Country'].nunique())
print("\nCountry List:\n", df['Country'].value_counts())

Unique Invoices: 53628
Unique StockCodes: 5305
Unique Descriptions: 5698
Unique Customers: 5942
Unique Countries: 43

Country List:
 Country
United Kingdom          981330
EIRE                     17866
Germany                  17624
France                   14330
Netherlands               5140
Spain                     3811
Switzerland               3189
Belgium                   3123
Portugal                  2620
Australia                 1913
Channel Islands           1664
Italy                     1534
Norway                    1455
Sweden                    1364
Cyprus                    1176
Finland                   1049
Austria                    938
Denmark                    817
Unspecified                756
Greece                     663
Japan                      582
Poland                     535
USA                        535
United Arab Emirates       500
Israel                     371
Hong Kong                  364
Singapore                  346
Malta                 

In [20]:
# Separate real products vs non-products
text_codes = df[~df['StockCode'].astype(str).str.match(r'^\d')]
print("Non-product rows:", len(text_codes))
print("\nWeird StockCodes:")
print(text_codes['StockCode'].value_counts())

Non-product rows: 6093

Weird StockCodes:
StockCode
POST         2122
DOT          1446
M            1421
C2            282
D             177
             ... 
DCGS0074        1
DCGS0073        1
DCGS0071        1
DCGS0066P       1
DCGS0067        1
Name: count, Length: 62, dtype: int64


In [21]:
print("Price Stats:\n", df['Price'].describe())
print("\nQuantity Stats:\n", df['Quantity'].describe())

# Check zero prices
print("\nZero price rows:", len(df[df['Price'] == 0]))

# Check negative quantities
print("Negative quantity rows:", len(df[df['Quantity'] < 0]))

# Check zero quantities
print("Zero quantity rows:", len(df[df['Quantity'] == 0]))

Price Stats:
 count    1.067371e+06
mean     4.649388e+00
std      1.235531e+02
min     -5.359436e+04
25%      1.250000e+00
50%      2.100000e+00
75%      4.150000e+00
max      3.897000e+04
Name: Price, dtype: float64

Quantity Stats:
 count    1.067371e+06
mean     9.938898e+00
std      1.727058e+02
min     -8.099500e+04
25%      1.000000e+00
50%      3.000000e+00
75%      1.000000e+01
max      8.099500e+04
Name: Quantity, dtype: float64

Zero price rows: 6202
Negative quantity rows: 22950
Zero quantity rows: 0


In [22]:
cancelled = df[df['Invoice'].astype(str).str.startswith('C')]
print("Cancellation rows:", len(cancelled))
print("\nSample cancellations:")
cancelled.head()

Cancellation rows: 19494

Sample cancellations:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia


In [23]:
# Check for weird descriptions
keywords = ['postage', 'fee', 'charge', 'bank', 'amazon', 'adjust', 'manual', 'cruk', 'dot']
weird_desc = df[df['Description'].str.lower().str.contains('|'.join(keywords), na=False)]
print("Weird description rows:", len(weird_desc))
print(weird_desc[['StockCode','Description']].value_counts().head(20))

Weird description rows: 37039
StockCode  Description                        
POST       POSTAGE                                2115
22386      JUMBO BAG PINK POLKADOT                1518
DOT        DOTCOM POSTAGE                         1444
M          Manual                                 1421
22384      LUNCH BAG PINK POLKADOT                1359
22646      CERAMIC STRAWBERRY CAKE MONEY BANK      970
22356      CHARLOTTE BAG PINK POLKADOT             965
22645      CERAMIC HEART FAIRY CAKE MONEY BANK     800
22301      COFFEE MUG CAT + BIRD DESIGN            706
22644      CERAMIC CHERRY CAKE MONEY BANK          664
22637      PIGGY BANK RETROSPOT                    662
22381      TOY TIDY PINK POLKADOT                  615
37370      RETRO COFFEE MUGS ASSORTED              579
22300      COFFEE MUG DOG + BALL DESIGN            573
22303      COFFEE MUG APPLES DESIGN                571
21216      SET 3 RETROSPOT TEA,COFFEE,SUGAR        535
21121      SET/10 RED POLKADOT PARTY CANDLE

In [24]:
# Full duplicate rows
print("Full duplicate rows:", df.duplicated().sum())

# Same invoice + same product = duplicate
partial_dup = df.duplicated(subset=['Invoice','StockCode'], keep=False)
print("Invoice+StockCode duplicates:", partial_dup.sum())

Full duplicate rows: 34335
Invoice+StockCode duplicates: 88585


# Removing Nulls and duplicates 

In [25]:
before = df.shape[0]
df.dropna(subset=['Customer ID'], inplace=True)
print(f"Removed {before - df.shape[0]} null Customer ID rows")
print("Shape now:", df.shape)

Removed 243007 null Customer ID rows
Shape now: (824364, 8)


In [26]:
before = df.shape[0]
df = df[~df['Invoice'].astype(str).str.startswith('C')]
print(f"Removed {before - df.shape[0]} cancellation rows")
print("Shape now:", df.shape)

Removed 18744 cancellation rows
Shape now: (805620, 8)


In [27]:
before = df.shape[0]
df = df[df['StockCode'].astype(str).str.match(r'^\d')]
print(f"Removed {before - df.shape[0]} non-product rows")
print("Shape now:", df.shape)

Removed 2927 non-product rows
Shape now: (802693, 8)


In [28]:
before = df.shape[0]
df = df[(df['Quantity'] > 0) & (df['Price'] > 0)]
print(f"Removed {before - df.shape[0]} negative/zero rows")
print("Shape now:", df.shape)

Removed 61 negative/zero rows
Shape now: (802632, 8)


In [29]:
before = df.shape[0]
df.drop_duplicates(inplace=True)
print(f"Removed {before - df.shape[0]} duplicate rows")
print("Shape now:", df.shape)

Removed 26055 duplicate rows
Shape now: (776577, 8)


In [30]:
# Fix date
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Fix Customer ID — remove decimal
df['Customer ID'] = df['Customer ID'].astype(int).astype(str)

# Fix Invoice — string
df['Invoice'] = df['Invoice'].astype(str)

print("Data types fixed ✅")
print(df.dtypes)

Data types fixed ✅
Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
Price                 float64
Customer ID            object
Country                object
dtype: object


In [31]:
# Revenue
df['Revenue'] = df['Quantity'] * df['Price']

# Time columns
df['Year']      = df['InvoiceDate'].dt.year
df['Month']     = df['InvoiceDate'].dt.month
df['Day']       = df['InvoiceDate'].dt.day
df['Hour']      = df['InvoiceDate'].dt.hour
df['DayOfWeek'] = df['InvoiceDate'].dt.day_name()
df['Quarter']   = df['InvoiceDate'].dt.quarter

print("New columns created ✅")
df.head()

New columns created ✅


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue,Year,Month,Day,Hour,DayOfWeek,Quarter
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4,2009,12,1,7,Tuesday,4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0,2009,12,1,7,Tuesday,4
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0,2009,12,1,7,Tuesday,4
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8,2009,12,1,7,Tuesday,4
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0,2009,12,1,7,Tuesday,4


In [32]:
print("=" * 50)
print("FINAL CLEANED DATA SUMMARY")
print("=" * 50)
print(f"Total Rows:         {df.shape[0]:,}")
print(f"Total Columns:      {df.shape[1]}")
print(f"Null Values:        {df.isnull().sum().sum()}")
print(f"Duplicate Rows:     {df.duplicated().sum()}")
print(f"Date Range:         {df['InvoiceDate'].min()} → {df['InvoiceDate'].max()}")
print(f"Total Revenue:      £{df['Revenue'].sum():,.2f}")
print(f"Unique Customers:   {df['Customer ID'].nunique():,}")
print(f"Unique Products:    {df['StockCode'].nunique():,}")
print(f"Unique Countries:   {df['Country'].nunique():,}")
print("=" * 50)

FINAL CLEANED DATA SUMMARY
Total Rows:         776,577
Total Columns:      15
Null Values:        0
Duplicate Rows:     0
Date Range:         2009-12-01 07:45:00 → 2011-12-09 12:50:00
Total Revenue:      £17,068,567.97
Unique Customers:   5,852
Unique Products:    4,619
Unique Countries:   41


In [33]:
df.to_csv('C:/Users/Yashwanth u/Desktop/Data_projects/retail-intelligence/cleaned/cleaned_retail.csv', index=False)
print("✅ Cleaned data saved!")

✅ Cleaned data saved!
